# Debug: Egg-box Likelihood with Uniform Prior

> **Debug status:** this migrated v3 notebook is excluded from the maintained, executed gallery. With 1,000 independent root lineages its evidence agreed with brute force, but the posterior mode weights remained visibly uneven. It needs a robust mode-occupation strategy before promotion.

This toy likelihood has many isolated posterior modes.

$L(x) = P(y | x) = (2. + \prod_i cos(\frac{\theta_i}{2})))^5$

and

$P(x) = \mathcal{U}[x \mid 0, 10\pi]$.


In [ ]:
from jax import config

# Needed because otherwise likelihoods hit numerical plateau
config.update("jax_enable_x64", True)

import matplotlib.pyplot as plt
import tensorflow_probability.substrates.jax as tfp
from jax import numpy as jnp
from jax import random, vmap
from jax.flatten_util import ravel_pytree

from jaxctx.priors.prior import Prior
from jaxns.core import NestedSampler
from jaxns.model import Model
from jaxns.diagnostics.reference import bruteforce_evidence

tfpd = tfp.distributions

In [ ]:


ndim = 2


def prior_model():
    theta = Prior(
        tfpd.Uniform(
            low=jnp.zeros(ndim),
            high=jnp.pi * 10 * jnp.ones(ndim),
        ),
        name='theta',
    ).realise()
    return jnp.power(2. + jnp.prod(jnp.cos(0.5 * theta)), 5)


model = Model(prior_model=prior_model)

log_Z_true = bruteforce_evidence(model=model, grid_res=250)
print(f"True log(Z)={log_Z_true}")


In [ ]:
u_example = model.sample_U(random.PRNGKey(0))
u_example_flat, unravel_fn = ravel_pytree(u_example)
u_vec = jnp.linspace(0., 1., 250, dtype=u_example_flat.dtype)
u_flat = jnp.stack([x.flatten() for x in jnp.meshgrid(*[u_vec] * model.U_ndims(), indexing='ij')], axis=-1)

# Evaluate the model log-likelihood over a grid in U-space
lik = vmap(lambda u: model.log_likelihood(unravel_fn(u)))(u_flat)
lik = lik.reshape((u_vec.size, u_vec.size))

plt.imshow(lik.T, origin='lower', extent=(0., 10., 0., 10.), cmap='jet')
plt.xlabel(r'$\theta_0 / \pi$')
plt.ylabel(r'$\theta_1 / \pi$')
plt.colorbar(label='log likelihood')
plt.show()

In [ ]:


# This experiment uses many independent roots because isolated modes can
# otherwise vanish. The remaining mode-weight imbalance is why this notebook
# stays in debug rather than the maintained example gallery.
nested_sampler = NestedSampler(
    model=model,
    root_allocation_degree=1_000,
    collect_phantom_samples=True,
)

state = nested_sampler.run(key=random.PRNGKey(42))
results = state.to_result().trim()


In [ ]:
# We can use the result method to display a summary
results.summary()

classic_evidence = results.sample_evidence_mc(
    num_samples=512,
    conditioning='classic',
    key=random.PRNGKey(43),
)
phantom_evidence = results.sample_evidence_mc(
    num_samples=512,
    conditioning='phantom',
    key=random.PRNGKey(44),
)
plt.hist(
    [classic_evidence.log_Z_samples, phantom_evidence.log_Z_samples],
    bins=30,
    density=True,
    histtype='step',
    label=['classic', 'phantom-conditioned'],
)
plt.axvline(log_Z_true, color='black', linestyle='dashed', label='brute force')
plt.xlabel(r'$\log Z$')
plt.ylabel('density')
plt.legend()
plt.show()

In [ ]:

# We plot useful diagnostics and a distribution cornerplot
results.plot_diagnostics()
results.plot_cornerplot()